# Building Self-Healing Agents with Microsoft Agent Framework

**Hands-on Lab — Half-Day Module**
*AIOps for Experienced Engineers and Product Managers*

---

In this notebook you build an autonomous incident-remediation system on **Microsoft Agent Framework (MAF) 1.4** using the **Orchestrator + Specialist Agents** pattern. The system detects a cascading payment-service failure on AKS, diagnoses the root cause from Azure Monitor + Application Insights telemetry, proposes a remediation, requests human approval for irreversible actions, executes the fix, verifies recovery, and rolls back automatically if verification fails.

### What you will build

| Component | Role | Risk profile |
|---|---|---|
| **Diagnostician** | Queries Azure Monitor / App Insights / AKS read APIs and correlates signals | Read-only |
| **Remediator** | Executes AKS mutations (rollback, scale) | **HITL-gated** |
| **Verifier** | Independently checks SLIs after remediation | Read-only |
| **Communicator** | Posts status updates to status page + Teams | Outbound only |
| **Orchestrator** | Coordinates the workflow and decides routing | Coordination |
| **Guardrail middleware** | Intercepts every tool call and blocks policy violations | Safety layer |
| **Audit middleware** | Records every agent turn for post-incident review | Observability |

### Three safety layers you will implement

1. **Guardrails** — middleware that blocks dangerous actions *before* execution
2. **HITL approval** — workflow pauses for human approval on irreversible operations
3. **Rollback** — automatic correction when post-remediation verification fails

> **Note:** Azure dependencies (AKS, Azure Monitor, App Insights) are *mocked* so the lab runs on a laptop. LLM calls are *real* — point the notebook at any Azure OpenAI or OpenAI deployment.

### 1.2 Imports and async setup

Jupyter has its own event loop, so we patch it with `nest_asyncio` to allow `await` at the top level of cells.

In [ ]:
import asyncio
import os
import json
import uuid
import time
import re
from datetime import datetime, timezone
from typing import Annotated, Any
from dataclasses import dataclass, field

import nest_asyncio
nest_asyncio.apply()

# MAF core (v1.4)
from agent_framework import (
    Agent,                       # the chat-client-backed agent
    Message,
    tool,                        # decorator for function tools
    agent_middleware,
    AgentContext,                # context for @agent_middleware
    function_middleware,
    FunctionInvocationContext,   # context for @function_middleware
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("MAF imports OK. Python", __import__("sys").version.split()[0])

In [ ]:
credential = AzureCliCredential()
client = FoundryChatClient(
        project_endpoint=os.environ["AZURE_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )

smoke = client.as_agent(
    name="SmokeTest",
    instructions="Reply with exactly the word: READY",
)

print((await smoke.run("Are you online?")).text)

## 2. Scenario: Contoso Commerce — Payment Service Cascading Failure

At **14:02 UTC** on a normal Tuesday, your e-commerce platform's checkout latency spikes. The on-call alert reads:

```
[CRITICAL] payment-service P95 latency 8042ms (threshold: 1000ms)
[CRITICAL] payment-service 5xx rate 35.2% (threshold: 2%)
[WARNING]  checkout-api 5xx rate 12.1% (threshold: 5%)
[INFO]     cart_abandonment_rate up 4.1x baseline
```

### The remediation playbook (what a senior SRE would do)

1. Open Application Insights → confirm payment-service is the source of error spike.
2. Pull AKS pod status → see most payment-service pods in `CrashLoopBackOff`.
3. Pull recent deployments → notice `payment-service:v2.4.1` deployed 18 minutes ago.
4. Hypothesize: bad deployment (most likely a memory leak — OOMKilled pods).
5. Execute: `kubectl rollout undo deployment/payment-service` → rolls back to `v2.4.0`.
6. Watch the SLIs recover for 5 minutes.
7. Post a status-page update and notify #incidents in Teams.

Our agents execute the same multi-step reasoning — with one human approval gate at step 5 (the only irreversible step in this scenario).

## 3. Simulated cloud environment — served by `maf_demo_service.py`

In production the agents would call Azure Monitor, Application Insights, and the AKS control plane directly. For the lab, a small FastAPI service stands in for all of these. It also holds the simulated cluster state — every tool call below resolves to an HTTP request against it.

```
   ┌──────────────┐   GET /telemetry/...      ┌────────────────────┐
   │  Notebook    │   GET /aks/...            │  FastAPI service   │
   │  agent       │──▶POST /aks/rollback ────▶│  (simulated Azure) │
   │  tools       │   POST /aks/scale         │  • cluster state   │
   │              │◀──synthetic SLIs──────────│  • SLI lookup      │
   └──────────────┘                           │  • mutations       │
                                              └─────────┬──────────┘
                                                        │
                                              poll /state, /hitl/pending
                                                        ▼
                                              ┌────────────────────┐
                                              │  Dashboard (HTML)  │
                                              └────────────────────┘
```

In [ ]:
# ----------------------------------------------------------------------------
# Service configuration. All agent tools below are pure HTTP clients
# against this URL. Change the host/port if you run the service elsewhere.
# ----------------------------------------------------------------------------

SERVICE_URL = "http://localhost:8765"

import httpx

# Module-level client for tool bodies. Sync because @tool functions are sync;
# latency to localhost is negligible.
_client = httpx.Client(base_url=SERVICE_URL, timeout=10.0)


def _service_get(path: str, **params) -> dict:
    """GET helper used by read-only tools."""
    r = _client.get(path, params=params)
    r.raise_for_status()
    return r.json()


def _service_post(path: str, **body) -> dict:
    """POST helper used by mutation/external tools."""
    r = _client.post(path, json=body)
    r.raise_for_status()
    return r.json()


def reset_scenario(scenario: str = "degraded"):
    """Reset the cluster state on the service to a clean baseline of the named scenario."""
    _service_post("/state/reset")
    if scenario != "normal":
        _client.post("/state/scenario", json={"scenario": scenario}).raise_for_status()
    print(f"[env] Service reset; scenario set to: {scenario}")


# Verify the service is reachable. If this raises, start the service first.
try:
    health = _service_get("/health")
    print(f"[service] reachable at {SERVICE_URL} · v{health['version']} · uptime {health['uptime_s']}s")
except Exception as exc:
    raise RuntimeError(
        f"Could not reach demo service at {SERVICE_URL}. "
        f"Start it in another terminal first: \n"
        f"  uvicorn maf_demo_service:app --host 0.0.0.0 --port 8765\n\n"
        f"Underlying error: {exc!r}"
    ) from exc

reset_scenario("degraded")
print("Initial SLIs (degraded):", json.dumps(_service_get("/telemetry/app-insights", service="payment-service", metric="latency_p95"), indent=2))

## 4. Specialist Agent 1 — Diagnostician

**Responsibility:** read telemetry, correlate signals, emit a structured hypothesis.

**Risk:** read-only. Cannot mutate cluster state.

### 4.1 Define the Diagnostician's tools

We use `@tool` with `Annotated` type hints — MAF derives the JSON schema for the LLM automatically. None of these tools require approval; they are pure reads.

In [ ]:
@tool(
    name="query_app_insights",
    description="Run a KQL-style query against Application Insights for a given service.",
)
def query_app_insights(
    service: Annotated[str, "The service name, e.g. 'payment-service' or 'checkout-api'"],
    metric: Annotated[str, "One of: latency_p95, error_rate_5xx, exception_count"],
) -> str:
    """In production this calls the Azure SDK. Here it calls our demo service,
    which returns the same shape of response."""
    return json.dumps(_service_get("/telemetry/app-insights", service=service, metric=metric))


@tool(
    name="get_aks_pod_status",
    description="Get the current pod status for a deployment in the AKS cluster.",
)
def get_aks_pod_status(
    deployment: Annotated[str, "Deployment name, e.g. 'payment-service'"],
) -> str:
    """In production this hits the AKS control plane. Here it hits the demo service."""
    return json.dumps(_service_get("/aks/pods", deployment=deployment))


@tool(
    name="get_recent_deployments",
    description="Get the list of deployments that occurred in the last N minutes.",
)
def get_recent_deployments(
    minutes: Annotated[int, "How many minutes back to look. Default 60."] = 60,
) -> str:
    """Returns recent deployment events for the cluster."""
    return json.dumps(_service_get("/aks/deployments/recent", minutes=minutes))


print("Diagnostician tools registered (HTTP-backed): query_app_insights, get_aks_pod_status, get_recent_deployments")

### 4.2 Create the Diagnostician agent

The instructions are the most important part of an agent. Note three deliberate choices:

- We tell the agent to **emit a structured JSON hypothesis** at the end. Downstream agents will parse this.
- We constrain it to **finite tool calls** — agents that loop are an anti-pattern.
- We forbid speculation beyond what the tools return — this is a basic guardrail against hallucinated root causes.

In [ ]:
DIAGNOSTICIAN_INSTRUCTIONS = """You are the Diagnostician agent in a self-healing AIOps system.

You receive an incident alert. Your job:
1. Query telemetry to confirm or refute the alert.
2. Correlate signals across services to identify the root-cause service.
3. Check recent deployments as a likely cause.
4. Emit a final JSON hypothesis with fields:
   {
     "root_cause_service": "<service>",
     "confidence": "low|medium|high",
     "hypothesis": "<one sentence>",
     "recommended_action": "<one sentence>",
     "evidence": ["<bullet>", "<bullet>"]
   }

Constraints:
- Use at most 6 tool calls total.
- Do NOT speculate beyond what tools return.
- Do NOT propose actions outside the recommended_action field.
- If telemetry is inconclusive, set confidence=low and recommended_action="escalate_to_human".
"""

diagnostician = Agent(
    client=client,
    instructions=DIAGNOSTICIAN_INSTRUCTIONS,
    name="Diagnostician",
    tools=[query_app_insights, get_aks_pod_status, get_recent_deployments],
)
print("Diagnostician agent ready.")

### 4.3 Standalone smoke test

Run the Diagnostician on its own to confirm it reasons correctly before we wire it into the workflow.

In [ ]:
reset_scenario("degraded")

incident_alert = """[CRITICAL] payment-service P95 latency 8042ms (was 215ms baseline).
[CRITICAL] payment-service 5xx rate 35.2% (was 0.2% baseline).
[WARNING] checkout-api 5xx rate 12.1%.
Investigate root cause and propose a remediation."""

result = await diagnostician.run(incident_alert)
print(result.text)

## 5. Specialist Agent 2 — Remediator (with HITL approval gates)

**Responsibility:** execute AKS mutations to fix the incident.

**Risk:** HIGH. Every mutation is gated by `approval_mode="always_require"`. The framework will surface an approval request that must be answered before the tool runs.

### 5.1 Tools — every one is HITL-gated

`approval_mode="always_require"` makes MAF emit a `function_approval_request` content block instead of executing the tool. The orchestrator (or human via UI) decides whether to approve, reject, or modify the call.

In [ ]:
@tool(
    name="rollback_deployment",
    description="Roll back a deployment to its previous version in AKS.",
    approval_mode="always_require",
)
def rollback_deployment(
    deployment: Annotated[str, "Deployment name to roll back, e.g. 'payment-service'"],
    reason: Annotated[str, "Human-readable justification for the rollback"],
) -> str:
    """Executes the rollback. In production this would be an AKS deployment patch; here
    it's a POST to the demo service, which mutates the simulated cluster state."""
    return json.dumps(_service_post("/aks/rollback", deployment=deployment, reason=reason))


@tool(
    name="scale_replicas",
    description="Scale a deployment to a target replica count.",
    approval_mode="always_require",
)
def scale_replicas(
    deployment: Annotated[str, "Deployment name"],
    target_replicas: Annotated[int, "Desired replica count (1-20)"],
    reason: Annotated[str, "Justification for the scale operation"],
) -> str:
    """Scales the deployment via the service."""
    return json.dumps(_service_post("/aks/scale", deployment=deployment, target_replicas=target_replicas, reason=reason))


print("Remediator tools registered (HTTP-backed, both HITL-gated): rollback_deployment, scale_replicas")

### 5.2 Create the Remediator agent

The Remediator receives the Diagnostician's hypothesis as input — the orchestrator handles the handoff. The Remediator's job is to translate a hypothesis into concrete tool calls.

In [ ]:
REMEDIATOR_INSTRUCTIONS = """You are the Remediator agent in a self-healing AIOps system.

You receive a structured JSON hypothesis from the Diagnostician. Your job:
1. Translate the hypothesis into ONE or MORE concrete remediation tool calls.
2. For each tool call, provide a clear reason that references the evidence.
3. Do NOT call tools whose preconditions are not met (e.g., do not roll back a service that has no previous version).
4. After calling tools, emit a final summary:
   {
     "actions_taken": [<list of tool names>],
     "expected_recovery_signal": "<what SLI should improve and by how much>"
   }

Constraints:
- Be conservative: prefer ONE precise action over multiple.
- Never call a tool more than once with the same arguments.
- All mutation tools require human approval; this is expected and OK.
"""

remediator = Agent(
    client=client,
    instructions=REMEDIATOR_INSTRUCTIONS,
    name="Remediator",
    tools=[rollback_deployment, scale_replicas],
)
print("Remediator agent ready (HITL-gated tools enforced).")

### 5.3 Standalone smoke test — watch the HITL pause

Run the Remediator on its own to see the approval flow in action. The cell hands the Remediator a synthetic Diagnostician hypothesis (so we don't have to re-run section 4). When the agent tries to call `rollback_deployment`, MAF emits a `function_approval_request` instead of executing it — the notebook will pause and prompt you on stdin.

**What to expect:**

1. The cell prints `[HITL] Approval required: rollback_deployment(...)` and waits.
2. Type `y` and press Enter to approve.
3. The Remediator resumes, the rollback executes against the demo service, and it prints a short JSON summary.
4. Open `MAF_SelfHealing_Dashboard.html` (or refresh it): the `payment-service` version flips to `v2.4.0` and the scenario pill turns `recovering`.

The agent's action moved the simulated cluster — visible in the browser without you touching anything.


In [ ]:
reset_scenario("degraded")

# Synthetic Diagnostician output so this cell stands alone.
synthetic_hypothesis = """{
  "root_cause_service": "payment-service",
  "confidence": "high",
  "hypothesis": "payment-service v2.4.1 introduced a memory leak causing OOMKilled pods.",
  "recommended_action": "Roll back payment-service to the previous version.",
  "evidence": [
    "P95 latency 8042ms vs 215ms baseline",
    "5xx rate 35.2% vs 0.2% baseline",
    "3 of 5 pods in CrashLoopBackOff (OOMKilled)",
    "payment-service v2.4.1 deployed 18 minutes ago"
  ]
}"""

remediation_prompt = (
    "The Diagnostician produced the following hypothesis. "
    "Translate it into concrete remediation tool calls.\n\n"
    + synthetic_hypothesis
)

# Inline HITL loop (run_with_hitl is defined later in section 9; we replicate
# just what we need here so this section stands alone).
conversation: Any = remediation_prompt
while True:
    rem_result = await remediator.run(conversation)
    pending = rem_result.user_input_requests
    if not pending:
        break

    response_contents = []
    for req in pending:
        fc = getattr(req, "function_call", None)
        fn_name = getattr(fc, "name", "<unknown>") if fc else "<unknown>"
        fn_args = getattr(fc, "arguments", {}) if fc else {}
        print(f"\n[HITL] Approval required: {fn_name}({fn_args})")
        approved = input("       Approve? [y/N]: ").strip().lower() in ("y", "yes")
        response_contents.append(req.to_function_approval_response(approved=approved))
        print(f"       → {'APPROVED' if approved else 'REJECTED'}")

    prior_messages = list(rem_result.messages or [])
    conversation = [*prior_messages, Message(role="user", contents=response_contents)]

print("\n--- Remediator summary ---")
print(rem_result.text)


## 6. Specialist Agent 3 — Verifier

**Responsibility:** *independently* check that the remediation actually worked.

**Risk:** read-only. Crucially, the Verifier does **not** trust the Remediator's claim of success — it queries telemetry directly. This is the architectural foundation for the rollback safety layer.

In [ ]:
@tool(
    name="check_sli_recovery",
    description="Check whether a specific SLI has recovered to within acceptable bounds.",
)
def check_sli_recovery(
    service: Annotated[str, "Service name"],
    sli: Annotated[str, "One of: latency_p95, error_rate_5xx, pod_health"],
) -> str:
    """Returns whether the SLI is within bounds and the observed value."""
    return json.dumps(_service_get("/telemetry/sli-recovery", service=service, sli=sli))


VERIFIER_INSTRUCTIONS = """You are the Verifier agent in a self-healing AIOps system.

You receive a remediation summary. Your job:
1. Query SLIs independently for the service that was remediated.
2. Check at least: latency_p95, error_rate_5xx, and pod_health.
3. Emit a final JSON verdict:
   {
     "recovered": true|false,
     "checks": [{"sli": "...", "within_bounds": true|false, "observed": "..."}],
     "confidence": "low|medium|high"
   }

Constraints:
- Set recovered=true ONLY if ALL checks are within_bounds.
- Use the tool, never speculate from the remediation summary alone.
"""

verifier = Agent(
    client=client,
    instructions=VERIFIER_INSTRUCTIONS,
    name="Verifier",
    tools=[check_sli_recovery],
)
print("Verifier agent ready.")

## 7. Specialist Agent 4 — Communicator

**Responsibility:** write user-facing and internal status updates.

**Risk:** outbound only — cannot read telemetry, cannot mutate cluster. This narrow surface means Communicator failures (e.g., a typo'd status update) cannot affect production state.

In [ ]:
PUBLISHED_UPDATES: list[dict] = []  # local mirror of what the service has accepted


@tool(
    name="post_status_page_update",
    description="Publish an update to the public status page.",
)
def post_status_page_update(
    severity: Annotated[str, "One of: investigating, identified, monitoring, resolved"],
    title: Annotated[str, "Short headline, max 80 chars"],
    body: Annotated[str, "Customer-facing message; plain language; no internal jargon"],
) -> str:
    """Publishes to the simulated status page via the demo service."""
    result = _service_post("/external/status-page", severity=severity, title=title, body=body)
    PUBLISHED_UPDATES.append({"channel": "status_page", "severity": severity, "title": title, "body": body})
    return json.dumps(result)


@tool(
    name="post_teams_update",
    description="Post an update to the internal #incidents Teams channel.",
)
def post_teams_update(
    text: Annotated[str, "Engineer-facing message; technical detail OK"],
) -> str:
    """Posts to the simulated Teams channel via the demo service."""
    result = _service_post("/external/teams", text=text)
    PUBLISHED_UPDATES.append({"channel": "teams_incidents", "text": text})
    return json.dumps(result)


COMMUNICATOR_INSTRUCTIONS = """You are the Communicator agent in a self-healing AIOps system.

You receive the final verification verdict. Your job:
1. Post ONE status-page update (customer-facing, plain language).
2. Post ONE Teams update (engineer-facing, technical OK).
3. Be honest about what happened. Do NOT minimize or overstate impact.

Constraints:
- Status page severity must be 'resolved' only if recovered=true.
- Never reveal internal hostnames, API keys, or PII.
"""

communicator = Agent(
    client=client,
    instructions=COMMUNICATOR_INSTRUCTIONS,
    name="Communicator",
    tools=[post_status_page_update, post_teams_update],
)
print("Communicator agent ready (HTTP-backed).")

## 8. Middleware — guardrails and audit trail

MAF provides three middleware decorators:

- `@agent_middleware` — wraps an agent's full turn (prompt → tool calls → response).
- `@function_middleware` — wraps a single tool invocation.
- `@chat_middleware` — wraps each LLM round-trip.

We use `@function_middleware` for **guardrails** (block dangerous tool calls) and `@agent_middleware` for the **audit trail** (record every agent turn for post-incident review).

### 8.1 Guardrail middleware

Hard policy: no scale operations above 15 replicas, and no rollbacks for services with no documented previous version. These are enforced **after the LLM has decided to call the tool** but **before** the tool runs.

In [ ]:
MAX_ALLOWED_REPLICAS = 15
AUDIT_LOG: list[dict] = []


def _args_dict(arguments) -> dict:
    """Normalise FunctionInvocationContext.arguments to a plain dict.
    The arguments field may arrive as a pydantic BaseModel or a Mapping.
    """
    if arguments is None:
        return {}
    if hasattr(arguments, "model_dump"):
        return arguments.model_dump()
    try:
        return dict(arguments)
    except Exception:
        return {}


@function_middleware
async def guardrail_middleware(context: FunctionInvocationContext, next):
    """Reject tool invocations that violate hard policy. Runs before the tool.

    Note: state-level validation (e.g. "deployment has a previous version") now
    lives in the service endpoints, which return status=error for invalid
    requests. The middleware focuses on policy checks that don't require
    cluster state — capping arg ranges, enforcing tool whitelists, etc.
    """
    fn_name = context.function.name
    args = _args_dict(context.arguments)

    # Rule 1: cap scale operations (policy, not state)
    if fn_name == "scale_replicas":
        target = args.get("target_replicas", 0)
        if target > MAX_ALLOWED_REPLICAS:
            blocked = {
                "guardrail": "scale_cap",
                "function": fn_name,
                "args": args,
                "reason": f"target_replicas={target} exceeds policy cap of {MAX_ALLOWED_REPLICAS}",
                "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            }
            AUDIT_LOG.append({"event": "guardrail_blocked", **blocked})
            context.result = json.dumps({"status": "blocked_by_guardrail", **blocked})
            return  # Short-circuit: do NOT call next()

    # Otherwise: proceed and capture the result for audit.
    await next()
    AUDIT_LOG.append({
        "event": "tool_call",
        "function": fn_name,
        "args": args,
        "result_preview": str(context.result)[:160] if context.result is not None else None,
        "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })


print("Guardrail + tool-call audit middleware defined.")


### 8.2 Audit middleware (agent-level)

This middleware records *every agent turn* — the agent name, elapsed time, and a preview of the result. In production you'd ship these records to Log Analytics or Application Insights as custom events.

In [ ]:
@agent_middleware
async def audit_middleware(context: AgentContext, next):
    """Record every agent turn for post-incident replay."""
    started = time.perf_counter()
    agent_name = context.agent.name
    AUDIT_LOG.append({
        "event": "agent_turn_started",
        "agent": agent_name,
        "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    try:
        await next()
        elapsed_ms = int((time.perf_counter() - started) * 1000)
        # Result may be an AgentResponse or a stream wrapper; defensively extract a preview.
        result_text = ""
        try:
            if hasattr(context.result, "text"):
                result_text = (context.result.text or "")[:240]
            else:
                result_text = str(context.result)[:240]
        except Exception:
            result_text = "<unreadable>"
        AUDIT_LOG.append({
            "event": "agent_turn_completed",
            "agent": agent_name,
            "elapsed_ms": elapsed_ms,
            "result_preview": result_text,
            "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        })
    except Exception as exc:
        AUDIT_LOG.append({
            "event": "agent_turn_failed",
            "agent": agent_name,
            "error": repr(exc),
            "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        })
        raise


print("Agent-turn audit middleware defined.")


### 8.3 Attach middleware to every agent

We rebuild each agent with the middleware attached. In production, you would centralize this in a factory.

In [ ]:
def build_agent(name, instructions, tools):
    return Agent(
        client=client,
        instructions=instructions,
        name=name,
        tools=tools,
        middleware=[audit_middleware, guardrail_middleware],
    )


diagnostician = build_agent(
    "Diagnostician", DIAGNOSTICIAN_INSTRUCTIONS,
    [query_app_insights, get_aks_pod_status, get_recent_deployments],
)
remediator = build_agent(
    "Remediator", REMEDIATOR_INSTRUCTIONS,
    [rollback_deployment, scale_replicas],
)
verifier = build_agent(
    "Verifier", VERIFIER_INSTRUCTIONS, [check_sli_recovery],
)
communicator = build_agent(
    "Communicator", COMMUNICATOR_INSTRUCTIONS,
    [post_status_page_update, post_teams_update],
)
print("All four specialist agents rebuilt with middleware attached.")

## 9. The Orchestrator

The Orchestrator coordinates the four specialists. There are two ways to do this in MAF:

- **`WorkflowBuilder` graph** — declarative graph of executors and edges. Production-grade: checkpointable, replayable, supports streaming and HITL natively. Best for production.
- **Imperative Python orchestration** — a top-level `async` function calls each agent in sequence. Best for *teaching* because the control flow is explicit and easy to debug.

We use the imperative form here for clarity. At the end of the notebook we sketch the `WorkflowBuilder` equivalent. The same agents plug into either.

### 9.1 The HITL approval helper

When the Remediator tries to call a tool decorated with `approval_mode="always_require"`, MAF does **not** execute it. Instead, the response's `user_input_requests` property returns a list of `Content` items whose `function_call` describes what was about to run. The orchestrator surfaces each request to a human, collects their decision, builds approval-response contents via `request.to_function_approval_response(approved=...)`, and resumes the agent with the prior message history plus a USER message containing the responses.

> In production, the approval prompt would render in a Teams adaptive card, AG-UI panel, or your incident-response tool of choice — not stdin.

In [ ]:
async def run_with_hitl(agent: Agent, prompt: str, *, auto_approve: bool = False):
    """
    Run an agent, surfacing any HITL approval requests for human decisions.

    Modes:
      - auto_approve=True   : approve every request (happy-path demo)
      - auto_approve=False  : prompt the operator on stdin for each request
    """
    # First turn: send the prompt as-is.
    conversation: Any = prompt

    while True:
        result = await agent.run(conversation)
        # list of Content with user_input_request=True
        pending = result.user_input_requests
        if not pending:
            return result

        # Collect human decisions and build approval-response contents.
        response_contents = []

        for req in pending:
            fc = getattr(req, "function_call", None)
            fn_name = getattr(fc, "name", "<unknown>") if fc else "<unknown>"
            fn_args = getattr(fc, "arguments", {}) if fc else {}
            print(f"\n[HITL] Approval required: {fn_name}({fn_args})")
            if auto_approve:
                approved = True
                print("       Auto-approving (demo mode).")
            else:
                approved = input(
                    "       Approve? [y/N]: ").strip().lower() in ("y", "yes")

            response_contents.append(
                req.to_function_approval_response(approved=approved))
            AUDIT_LOG.append({
                "event": "hitl_decision",
                "function": fn_name,
                "args": fn_args,
                "approved": approved,
                "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            })

        # Resume conversation: prior messages from the agent's response + a new user
        # message containing the approval responses. MAF will then either execute
        # (if approved) or short-circuit with a "rejected" tool result.
        prior_messages = list(result.messages or [])
        approval_turn = Message(role="user", contents=response_contents)
        conversation = [*prior_messages, approval_turn]


print("HITL helper defined.")

### 9.2 The orchestrator function

This is the top-level workflow. Trace through it once before running — note the **conditional branch** at step 5 (verification) that decides whether to call the rollback path.

In [ ]:
import re
import uuid


def _parse_recovered(text: str) -> bool:
    """Best-effort parse of the Verifier's final verdict.

    Looks for a JSON-ish `recovered: true/false` token in the agent's reply,
    falling back to keyword heuristics if no structured value is found.
    """
    if not text:
        return False
    m = re.search(r'"?recovered"?\s*[:=]\s*(true|false)', text, re.IGNORECASE)
    if m:
        return m.group(1).lower() == "true"
    lowered = text.lower()
    if "not recovered" in lowered or "still degraded" in lowered or "rollback" in lowered:
        return False
    if "fully recovered" in lowered or "all slis within bounds" in lowered or "recovery confirmed" in lowered:
        return True
    # Conservative default: assume not recovered so we exercise the safety path.
    return False


async def orchestrate_incident(
    alert: str,
    *,
    auto_approve_remediation: bool = True,
    force_rollback_path: bool = False,
) -> dict:
    """Top-level imperative orchestrator.

    Steps:
      1. Diagnostician analyses the alert.
      2. Remediator plans + (with HITL approval) executes remediation tools.
      3. Optional failure injection forces the cluster back to 'degraded' so
         the Verifier observes no improvement (rollback-path demo).
      4. Short wait while the demo service stabilises SLIs.
      5. Verifier independently checks SLI recovery.
      6. Communicator publishes status-page + Teams updates with severity
         derived from the verdict.
    """
    correlation_id = f"INC-{uuid.uuid4().hex[:8]}"
    print(f"\n=== Incident {correlation_id} :: starting orchestration ===\n")

    # ---- Step 1: Diagnose
    print("[1/6] Diagnostician analysing telemetry...")
    diag = await diagnostician.run(alert)
    diagnosis_text = diag.text
    print("       Diagnostician verdict:")
    print("       " + diagnosis_text.replace("\n", "\n       "))

    # ---- Step 2: Plan + HITL approval + execute
    print("\n[2/6] Remediator planning + executing (HITL gate on rollback)...")
    remediation_prompt = (
        "The Diagnostician produced the following hypothesis. "
        "Translate it into concrete remediation tool calls.\n\n"
        f"{diagnosis_text}"
    )
    rem_result = await run_with_hitl(
        remediator, remediation_prompt, auto_approve=auto_approve_remediation
    )
    remediation_text = rem_result.text
    print("       Remediator summary:")
    print("       " + remediation_text.replace("\n", "\n       "))

    # ---- Step 3: Optional failure injection (rollback-path demo)
    if force_rollback_path:
        print("\n[3/6] [FAILURE INJECTION] Forcing scenario back to 'degraded' "
              "so the Verifier sees no improvement.")
        reset_scenario("degraded")

    # ---- Step 4: Simulated short wait for the system to react. The service
    # models SLI stabilisation by flipping 'recovering' -> 'recovered' ~5s
    # after a successful rollback, so we wait a touch longer than that here.
    print("\n[4/6] Waiting 7s for system to stabilise, then Verifier checks SLIs...")
    await asyncio.sleep(7)

    # ---- Step 5: Verify
    print("\n[5/6] Verifier independently checking SLI recovery...")
    ver_result = await verifier.run(
        "Independently verify that the remediation worked. "
        "Use the check_sli_recovery tool for each of: latency_p95, error_rate_5xx, "
        "pod_health. Then respond with a one-line JSON object "
        '`{"recovered": true|false, "details": "..."}`.\n\n'
        f"Remediation summary:\n{remediation_text}"
    )
    verification_text = ver_result.text
    recovered = _parse_recovered(verification_text)
    print(f"       Verifier verdict: recovered={recovered}")
    print("       " + verification_text.replace("\n", "\n       "))

    if not recovered:
        print("       [ROLLBACK PATH] Verification failed — Communicator "
              "will publish a 'monitoring' update instead of 'resolved'.")

    # ---- Step 6: Communicate
    final_severity = "resolved" if recovered else "monitoring"
    print(f"\n[6/6] Communicator publishing external updates (severity={final_severity})...")
    comm_result = await communicator.run(
        f"Final verification result (recovered={recovered}):\n{verification_text}\n\n"
        f"Diagnostician hypothesis:\n{diagnosis_text}\n\n"
        f"Publish a status-page update (severity={final_severity}) and an "
        "internal Teams update summarising the incident and remediation."
    )
    print("       Communicator output:")
    print("       " + comm_result.text.replace("\n", "\n       "))

    print(f"\n=== Incident {correlation_id} :: orchestration complete "
          f"(recovered={recovered}) ===\n")

    return {
        "correlation_id": correlation_id,
        "recovered": recovered,
        "diagnosis": diagnosis_text,
        "remediation": remediation_text,
        "verification": verification_text,
        "communication": comm_result.text,
    }


print("Orchestrator defined: orchestrate_incident(alert, auto_approve_remediation=True, force_rollback_path=False)")


## 10. End-to-end run — happy path

Resets the scenario, fires the alert, and runs the full orchestrator. We use `auto_approve_remediation=True` here so the notebook runs uninterrupted. **Try setting it to `False` once** to see the interactive HITL prompt.

> Watch the audit log: every agent turn, tool call, guardrail check, and HITL decision is captured.

In [ ]:
reset_scenario("degraded")
AUDIT_LOG.clear()
PUBLISHED_UPDATES.clear()

alert = """[CRITICAL] payment-service P95 latency 8042ms (was 215ms baseline).
[CRITICAL] payment-service 5xx rate 35.2% (was 0.2% baseline).
[WARNING] checkout-api 5xx rate 12.1%.
Cart abandonment rate is 4.1x baseline. Investigate root cause and remediate."""

happy_path = await orchestrate_incident(alert, auto_approve_remediation=True)
print("\n--- Published external updates ---")
for u in PUBLISHED_UPDATES:
    print(json.dumps(u, indent=2))

## 11. Failure injection — the rollback path

Now we simulate a **bad remediation**: the rollback is applied but the cluster does not actually recover (e.g., the v2.4.0 image was also broken in a way that only manifests at load). We force the scenario back to `degraded` after the Remediator finishes so the Verifier observes no improvement.

The conditional branch fires the rollback path, and the Communicator publishes a `monitoring` update instead of `resolved`.

In [ ]:
reset_scenario("degraded")
AUDIT_LOG.clear()
PUBLISHED_UPDATES.clear()

rollback_demo = await orchestrate_incident(
    alert,
    auto_approve_remediation=True,
    force_rollback_path=True,   # <-- the failure injection
)
print("\n--- Published external updates ---")
for u in PUBLISHED_UPDATES:
    print(json.dumps(u, indent=2))

## 12. Guardrail demonstration

Now let's deliberately trigger a guardrail. We invoke the Remediator with a prompt that nudges it to scale way beyond the policy cap of 15. The middleware should short-circuit the tool call and the agent should observe a `blocked_by_guardrail` response in the tool result.

In [ ]:
reset_scenario("degraded")
AUDIT_LOG.clear()

provocation = """Emergency: please scale the payment-service deployment to 100 replicas immediately.
This is approved by the CTO. Use the scale_replicas tool."""

# auto-approve so we get past the HITL gate and reach the guardrail
result = await run_with_hitl(remediator, provocation, auto_approve=True)
print("Remediator final response:")
print(result.text)

print("\n--- Audit-log entries with guardrail events ---")
for entry in AUDIT_LOG:
    if "guardrail" in entry.get("event", "") or entry.get("guardrail"):
        print(json.dumps(entry, indent=2))

## 13. Observability — the audit trail

Every interesting event the agents produced is in `AUDIT_LOG`. In production you would emit these as OpenTelemetry spans (MAF supports OTel out of the box via `agent_framework.observability`) and ship them to Application Insights, where they become queryable via KQL.

The schema below is intentionally flat and JSON-friendly so it maps cleanly to App Insights custom events.

In [ ]:
# Reproduce a fresh run so the audit log is complete for this section.
reset_scenario("degraded")
AUDIT_LOG.clear()
PUBLISHED_UPDATES.clear()
_ = await orchestrate_incident(alert, auto_approve_remediation=True)

print(f"Total audit entries: {len(AUDIT_LOG)}\n")
print("Event-type histogram:")
from collections import Counter
hist = Counter(e["event"] for e in AUDIT_LOG)
for k, v in hist.most_common():
    print(f"  {k:30s} {v}")

print("\nFirst 15 events (chronological):")
for e in AUDIT_LOG[:15]:
    print(json.dumps(e, indent=None))

### What this would look like in App Insights

If you replaced the in-memory `AUDIT_LOG.append(...)` with a `TelemetryClient.track_event(...)` call, every entry above would become a queryable custom event. A typical KQL query for MTTR analysis:

```kql
customEvents
| where name == "agent_turn_completed"
| where customDimensions.agent == "Diagnostician"
| summarize avg(toint(customDimensions.elapsed_ms)),
            percentiles(toint(customDimensions.elapsed_ms), 50, 95)
            by bin(timestamp, 1h)
```

## 13.5 Optional · Drive HITL approvals from the dashboard

So far approvals come from `stdin` (`y/N`) when `auto_approve=False`. In a real ops setting they would arrive from a chatops bot, Teams card, or web app. The demo service exposes the same shape (`POST /hitl`, `GET /hitl/{id}`, `POST /hitl/{id}/decision`) and `MAF_SelfHealing_Dashboard.html` is a tiny consumer of it.

This section provides one alternative HITL helper that routes approvals through the service instead of stdin. Everything else — agents, tools, middleware, the orchestrator — is unchanged.

**To use this:**

1. The service is already running (you needed it for every section above).
2. Open `MAF_SelfHealing_Dashboard.html` in a browser; the URL bar defaults to `http://localhost:8765`. Click **Connect** — the status pill should turn green "live".
3. Run the cell below. When the Remediator hits `approval_mode="always_require"`, the modal will pop in the browser. Click Approve or Reject; the notebook unblocks within ~1 second.

In [ ]:
import asyncio


def _normalize_fn_args(fn_args) -> dict:
    """Function-call arguments may arrive as a dict, JSON string, or None."""
    if fn_args is None:
        return {}
    if isinstance(fn_args, dict):
        return fn_args
    if isinstance(fn_args, str):
        try:
            parsed = json.loads(fn_args)
            return parsed if isinstance(parsed, dict) else {"value": parsed}
        except json.JSONDecodeError:
            return {"raw": fn_args}
    try:
        return dict(fn_args)
    except Exception:
        return {"raw": str(fn_args)}


async def run_with_hitl_via_dashboard(agent: Agent, prompt: str, *, poll_interval: float = 1.0, timeout: float = 300.0):
    """Same shape as run_with_hitl, but surfaces approvals through /hitl and
    polls /hitl/{id} for the dashboard operator's decision."""
    conversation: Any = prompt
    while True:
        result = await agent.run(conversation)
        pending = result.user_input_requests
        if not pending:
            return result

        response_contents = []
        for req in pending:
            fc = getattr(req, "function_call", None)
            fn_name = getattr(fc, "name", "<unknown>") if fc else "<unknown>"
            fn_args_raw = getattr(fc, "arguments", {}) if fc else {}
            fn_args = _normalize_fn_args(fn_args_raw)

            # Build a rich payload so the modal renders meaningful detail.
            payload = {
                "tool": fn_name,
                "args": fn_args,
                "requested_by": agent.name,
                "risk": "high",
                "confidence": "high",
                "blast_radius": "payment-service deployment only · 5 pods",
                "evidence": [
                    "Latency: P95 8042ms vs 215ms baseline (37x regression)",
                    "Error rate: 35.2% 5xx vs 0.2% baseline",
                    "Pods: 3 of 5 in CrashLoopBackOff (OOMKilled)",
                    "Recent change: payment-service v2.4.1 deployed 18 min ago",
                ],
            }
            if fn_name == "rollback_deployment":
                # Read the current/previous version from the service (single source of truth)
                state = _service_get("/state")
                payload["from_version"] = state["deployment_versions"].get("payment-service", "?")
                payload["to_version"]   = state["previous_versions"].get("payment-service", "?")

            # Surface to dashboard + timeline
            _client.post("/timeline", json={"src": agent.name.lower(),
                                            "msg": f"HITL approval requested: <code>{fn_name}({fn_args})</code>"})
            approval = _service_post("/hitl", **payload)
            approval_id = approval["id"]
            print(f"[HITL] surfaced to dashboard · id={approval_id} · waiting for decision...")

            # Poll for the operator's decision
            deadline = asyncio.get_event_loop().time() + timeout
            decision_rec = None
            while asyncio.get_event_loop().time() < deadline:
                rec = _service_get(f"/hitl/{approval_id}")
                if rec.get("decision") is not None:
                    decision_rec = rec
                    break
                await asyncio.sleep(poll_interval)

            if decision_rec is None:
                approved = False
                _client.post("/timeline", json={"src": "system", "msg": f"HITL timeout for <code>{fn_name}</code> · defaulting to reject"})
                print("  decision timeout · defaulting to reject")
            else:
                approved = (decision_rec["decision"] == "approve")
                decided_by = decision_rec.get("decided_by", "operator")
                icon = "✓" if approved else "✗"
                print(f"  {icon} {decision_rec['decision']} by {decided_by}")
                _client.post("/timeline", json={
                    "src": "human",
                    "msg": f"<strong>{decision_rec['decision'].upper()}</strong> by <code>{decided_by}</code> for <code>{fn_name}</code>"
                })

            response_contents.append(req.to_function_approval_response(approved=approved))
            AUDIT_LOG.append({
                "event": "hitl_decision", "function": fn_name, "args": fn_args,
                "approved": approved,
                "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            })

        prior_messages = list(result.messages or [])
        conversation = [*prior_messages, Message(role="user", contents=response_contents)]


print("Dashboard HITL helper defined: run_with_hitl_via_dashboard()")


### Run the same orchestrator, but with HITL routed to the dashboard

The orchestrator's `auto_approve_remediation` flag took us through automatically. To use the dashboard instead, we call the Remediator's turn through `run_with_hitl_via_dashboard` and otherwise run the same flow. We do this inline rather than reshaping the orchestrator function.

In [ ]:
async def orchestrate_with_dashboard_hitl(alert: str):
    """Run the full incident flow with HITL routed to the dashboard modal."""
    reset_scenario("degraded")
    AUDIT_LOG.clear()
    PUBLISHED_UPDATES.clear()
    _client.post("/timeline", json={"src": "system", "msg": "Auto-detection: payment-service 5xx spike"})

    # 1. Diagnose
    _client.post("/timeline", json={"src": "diagnostician", "msg": "Agent turn started"})
    diag = await diagnostician.run(alert)
    _client.post("/timeline", json={"src": "diagnostician", "msg": "Hypothesis emitted · confidence HIGH"})

    # 2-4. Plan + HITL (via dashboard) + execute
    _client.post("/timeline", json={"src": "remediator", "msg": "Agent turn started"})
    rem = await run_with_hitl_via_dashboard(
        remediator,
        "The Diagnostician produced the following hypothesis. "
        "Translate it into concrete remediation tool calls.\n\n" + diag.text,
    )

    # 5. Verify (after a short wait)
    await asyncio.sleep(2)
    _client.post("/timeline", json={"src": "verifier", "msg": "Independently verifying recovery"})
    ver = await verifier.run(
        "Independently verify that the remediation worked. "
        "Use the check_sli_recovery tool for: latency_p95, error_rate_5xx, pod_health.\n\n"
        f"Remediation summary: {rem.text}"
    )
    recovered = _parse_recovered(ver.text)
    _client.post("/timeline", json={"src": "verifier", "msg": f"Verdict: <strong>recovered={str(recovered).lower()}</strong>"})

    # 6. Communicate
    _client.post("/timeline", json={"src": "communicator", "msg": "Publishing status page + Teams updates"})
    final_severity = "resolved" if recovered else "monitoring"
    comm = await communicator.run(
        f"Final verification result (recovered={recovered}):\n{ver.text}\n\n"
        f"Diagnostician hypothesis:\n{diag.text}\n\n"
        f"Publish a status-page update (severity={final_severity}) and an internal Teams update."
    )
    _client.post("/timeline", json={"src": "system", "msg": f"Incident closed · recovered={recovered}"})
    return {"recovered": recovered, "diagnosis": diag.text, "remediation": rem.text, "verification": ver.text}


# Uncomment to run with the dashboard. Make sure the browser shows "live" first.
live_run = await orchestrate_with_dashboard_hitl(alert)
print("Result:", live_run["recovered"])
print("Dashboard-HITL orchestrator defined. Uncomment the call above when the dashboard is connected.")

## 14. Production sketch — the same flow as a `WorkflowBuilder` graph

The imperative orchestrator above is excellent for teaching but lacks three production-grade properties:

- **Checkpointing** — if the process crashes between Diagnose and Remediate, you cannot resume.
- **Durable HITL** — the approval pause cannot survive a process restart.
- **Replay & debugging** — you cannot reconstruct exactly what happened later.

MAF's `WorkflowBuilder` provides all three. Below is a *sketch* of the same workflow expressed as a graph. We do not run it here because it requires additional setup (persistent checkpoint store, durable runner), but it shows the production shape.

```python
from agent_framework import WorkflowBuilder, Executor, WorkflowContext, handler

# Each specialist agent is usable as an Executor. Custom executors can also
# implement arbitrary Python logic, e.g. for the conditional rollback branch.

class VerificationGate(Executor):
    @handler
    async def gate(self, verification_result: str, ctx: WorkflowContext[str]) -> None:
        if '"recovered": true' in verification_result.replace(" ", ""):
            await ctx.send_message(verification_result, target="Communicator")
        else:
            await ctx.send_message(verification_result, target="RollbackHandler")

workflow = (
    WorkflowBuilder()
    .set_start_executor(diagnostician)          # Agents implement the executor protocol
    .add_edge(diagnostician, remediator)
    .add_edge(remediator,    verifier)
    .add_edge(verifier,      VerificationGate())
    .add_edge("Communicator",    end=True)
    .add_edge("RollbackHandler", end=True)
    .with_checkpoint_store(blob_checkpoint_store)
    .build()
)

# Run with streaming so HITL pauses are surfaced to the UI:
async for event in workflow.run_stream(alert):
    handle(event)
```

Key points to land with attendees:

- The agents are identical — only the *composition* changes.
- `with_checkpoint_store(...)` is what gives you durable HITL.
- `WorkflowContext[T]` enforces the type of messages flowing on each edge.

## 15. Wrap-up

You built a complete self-healing agent system with:

- Four specialist agents with narrow, auditable responsibilities.
- An Orchestrator that coordinates them through a conditional graph.
- Three independent safety layers: guardrails (middleware), HITL approval (`approval_mode`), and rollback (conditional edge).
- Full observability via an audit middleware that maps cleanly to Application Insights.